# Lab 14 — Production AI Pipelines

Operationalize Cortex AI using Snowflake's pipeline primitives: Dynamic Tables, Streams, and Tasks.

| Item | Detail |
|---|---|
| **Features** | Dynamic Tables, Streams, Tasks + Cortex AI |
| **Exam Domain** | 2.0 Gen AI & LLM Functions (production patterns) |
| **You'll learn** | Incremental AI pipelines, scheduled enrichment, auto-refresh |

**What You'll Learn**

- Dynamic Tables with Cortex AI enrichment
- Streams for change detection
- Tasks for scheduled AI processing
- Event-driven AI with `SYSTEM$STREAM_HAS_DATA`

## Step 1 — Environment Setup

> **AI SQL Functions** — This lab uses the preferred `AI_` prefixed functions. 
Equivalent legacy functions: `AI_CLASSIFY` (replaces `SNOWFLAKE.CORTEX.CLASSIFY_TEXT`), `AI_SENTIMENT` (replaces `SNOWFLAKE.CORTEX.SENTIMENT`), `AI_SUMMARIZE_AGG` (replaces `SNOWFLAKE.CORTEX.SUMMARIZE`).

In [ ]:
USE ROLE DS_ROLE;
USE WAREHOUSE DS_WH;

-- Database pre-created by SYSADMIN in 00-admin-setup
USE DATABASE GENAI_LEARNING;
USE SCHEMA PUBLIC;

## Step 2 — Dynamic Tables with Cortex AI

Dynamic tables automatically refresh when source data changes.
Add Cortex AI functions to create self-maintaining AI-enriched tables.


In [ ]:
CREATE OR REPLACE TABLE support_tickets (
    ticket_id INT AUTOINCREMENT,
    subject VARCHAR(500),
    body TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

INSERT INTO support_tickets (subject, body) VALUES
('Dashboard not loading', 'The analytics dashboard has been showing a blank screen since this morning. Multiple users affected. This is blocking our daily standup.'),
('Slow query performance', 'Our nightly ETL job is taking 3x longer than usual. The main aggregation query on the sales table went from 5 minutes to 15 minutes.'),
('Access denied error', 'New team member cannot access the ANALYTICS schema. Getting permission denied on SELECT. They have the ANALYST role assigned.');

In [ ]:
CREATE OR REPLACE DYNAMIC TABLE enriched_tickets
    WAREHOUSE = DS_WH
    TARGET_LAG = '1 minute'
    AS
SELECT
    ticket_id, subject, body, created_at,
    AI_CLASSIFY(
        body,
        ['performance', 'access_control', 'bug', 'feature_request', 'data_issue']
    ):labels[0]::STRING AS category,
    AI_SENTIMENT(body):categories[0]:sentiment::STRING AS urgency_score,
    AI_SUMMARIZE_AGG(body) AS auto_summary
FROM support_tickets;

In [ ]:
SELECT * FROM enriched_tickets;

## Step 3 — Streams for Change Detection

Streams track changes to tables. Combined with tasks, they enable event-driven AI processing.

In [ ]:
CREATE OR REPLACE STREAM ticket_changes ON TABLE support_tickets;

In [ ]:
INSERT INTO support_tickets (subject, body) VALUES
('Export feature broken', 'CSV export from the reports page is generating empty files. Started after the last deployment on Friday.');

In [ ]:
SELECT * FROM ticket_changes;

## Step 4 — Tasks for Scheduled Processing

Tasks run on a schedule or when triggered by stream data.

In [ ]:
CREATE OR REPLACE TASK process_new_tickets
    WAREHOUSE = DS_WH
    SCHEDULE = '5 MINUTE'
    WHEN SYSTEM$STREAM_HAS_DATA('ticket_changes')
AS
    INSERT INTO enriched_tickets (ticket_id, subject, body, created_at, category, urgency_score, auto_summary)
    SELECT
        ticket_id, subject, body, created_at,
        AI_CLASSIFY(body, ['performance', 'access_control', 'bug', 'feature_request', 'data_issue']):labels[0]::STRING,
        AI_SENTIMENT(body):categories[0]:sentiment::STRING,
        AI_SUMMARIZE_AGG(body)
    FROM ticket_changes
    WHERE METADATA$ACTION = 'INSERT';

## Key Takeaways

- **Dynamic Tables** + Cortex AI = self-maintaining enriched tables.
- **Streams** detect changes; **Tasks** process them on schedule.
- Use `WHEN SYSTEM$STREAM_HAS_DATA()` for event-driven AI processing.
- `TARGET_LAG` on Dynamic Tables controls refresh frequency.
- These patterns enable production-grade AI pipelines entirely in SQL.
